# Session 4: Day 1 Lab Review & Integration (Instructor Notebook)

This capstone session integrates all Day 1 concepts — LLM foundations (Session 1), prompt engineering (Session 2), and model evaluation (Session 3) — into practical consulting AI exercises.

**Instructor version** — contains 3 guided demos + 4 tasks with complete solutions and approach explanations.

## Learning Objectives

By the end of this session, you will be able to:

1. Combine prompting, API usage, and evaluation into end-to-end consulting AI workflows
2. Design a multi-component consulting AI system using Day 1 techniques
3. Evaluate AI-generated consulting deliverables against McKinsey quality standards
4. Estimate costs and make model selection decisions for consulting use cases
5. Preview how Day 2 frameworks (LangChain, LangGraph) build on these foundations

In [17]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

import openai
import json
import os
import time
import numpy as np

client = openai.OpenAI()
print(f"Model: {os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')}")
print("Setup complete.")

Model: gpt-4o-mini
Setup complete.


---
## Demo 1: End-to-End Consulting Analysis Pipeline

This demo combines API calls (Session 1), structured prompting (Session 2), and automated evaluation (Session 3) into a single pipeline that generates and quality-checks a consulting deliverable.

In [18]:
# Demo 1: End-to-End Consulting Analysis Pipeline
# Combines: API calls (S1) + Structured prompting (S2) + G-Eval scoring (S3)

def g_eval_quick(question, response, criterion, description):
    """Lightweight G-Eval scorer (from Session 3)."""
    prompt = f"""Rate this consulting response on {criterion} (1-5).
Criterion: {description}
Question: {question}
Response: {response}
Return JSON with 'score' (1-5) and 'reasoning' (one sentence)."""
    result = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        temperature=0, max_tokens=200
    )
    return json.loads(result.choices[0].message.content)


def consulting_pipeline(question, persona='McKinsey senior consultant'):
    """Generate a consulting analysis, then auto-evaluate it."""
    print(f"Question: {question}")
    print(f"Persona: {persona}")
    print("-" * 50)
    
    # Step 1: Generate (Session 1 + Session 2 techniques)
    start = time.time()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': f'You are a {persona}. Provide structured, MECE, data-driven analysis with specific recommendations.'},
            {'role': 'user', 'content': question}
        ],
        temperature=0, max_tokens=500
    )
    latency = round((time.time() - start) * 1000)
    content = response.choices[0].message.content
    tokens = response.usage.total_tokens
    
    print(f"\nGenerated Response ({latency}ms, {tokens} tokens):")
    print(content[:300] + '...' if len(content) > 300 else content)
    
    # Step 2: Evaluate (Session 3 techniques)
    criteria = {
        'MECE Structure': 'Response breaks down the problem into mutually exclusive, collectively exhaustive categories.',
        'Actionability': 'Recommendations are specific, prioritized, and implementable.',
        'Executive Readiness': 'Output is polished enough for a C-suite presentation.'
    }
    
    print(f"\nQuality Evaluation:")
    total = 0
    for crit_name, crit_desc in criteria.items():
        result = g_eval_quick(question, content, crit_name, crit_desc)
        score = result.get('score', 0)
        total += score
        bar = '\u2588' * score + '\u2591' * (5 - score)
        print(f"  {crit_name:20s} {bar} {score}/5")
    
    avg = round(total / len(criteria), 1)
    verdict = 'PARTNER-READY' if avg >= 4.0 else 'NEEDS REVISION' if avg >= 3.0 else 'REWORK'
    print(f"\n  Overall: {avg}/5.0 | Verdict: {verdict}")
    return {'content': content, 'avg_score': avg, 'verdict': verdict, 'latency_ms': latency, 'tokens': tokens}


# Run the pipeline
result = consulting_pipeline(
    'What are the key value creation levers for a PE-backed healthcare services platform?'
)


Question: What are the key value creation levers for a PE-backed healthcare services platform?
Persona: McKinsey senior consultant
--------------------------------------------------

Generated Response (7797ms, 549 tokens):
To analyze the key value creation levers for a Private Equity (PE)-backed healthcare services platform, we can structure our analysis into three main categories: Operational Efficiency, Revenue Growth, and Strategic Positioning. Each category will be broken down into specific levers that are mutuall...

Quality Evaluation:
  MECE Structure       █████ 5/5
  Actionability        ████░ 4/5
  Executive Readiness  ████░ 4/5

  Overall: 4.3/5.0 | Verdict: PARTNER-READY


---
## Demo 2: Cost-Quality Trade-off Analysis

For production consulting AI, we need to balance quality against cost and latency. This demo runs the same query at different configurations and plots the trade-off to find the optimal operating point.

In [19]:
# Demo 2: Cost-Quality Trade-off Analysis
# Combines: Cost estimation (S1) + Temperature comparison (S1) + Evaluation (S3)

import pandas as pd

PRICING = {
    'gpt-4o-mini': {'input': 0.15 / 1_000_000, 'output': 0.60 / 1_000_000},
    'gpt-4o': {'input': 2.50 / 1_000_000, 'output': 10.00 / 1_000_000},
}

question = 'How should a mid-size European bank approach digital transformation to improve customer acquisition?'
configs = [
    {'temperature': 0.0, 'max_tokens': 200, 'label': 'Fast/Cheap (T=0, 200 tok)'},
    {'temperature': 0.0, 'max_tokens': 500, 'label': 'Standard (T=0, 500 tok)'},
    {'temperature': 0.7, 'max_tokens': 500, 'label': 'Creative (T=0.7, 500 tok)'},
]

results = []
for config in configs:
    start = time.time()
    resp = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': 'You are a McKinsey digital strategy expert.'},
            {'role': 'user', 'content': question}
        ],
        temperature=config['temperature'],
        max_tokens=config['max_tokens']
    )
    latency = round((time.time() - start) * 1000)
    content = resp.choices[0].message.content
    usage = resp.usage
    
    # Score
    score = g_eval_quick(question, content, 'Overall Quality',
        'Response is structured, actionable, and ready for executive review.')['score']
    
    # Cost
    model_name = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
    cost = 0
    if model_name in PRICING:
        cost = usage.prompt_tokens * PRICING[model_name]['input'] + usage.completion_tokens * PRICING[model_name]['output']
    
    results.append({
        'Config': config['label'],
        'Quality': f"{score}/5",
        'Latency': f"{latency}ms",
        'Tokens': usage.total_tokens,
        'Cost': f"${cost:.6f}",
        'Response Length': len(content),
    })

df = pd.DataFrame(results)
print('COST-QUALITY TRADE-OFF ANALYSIS')
print('=' * 70)
print(df.to_string(index=False))
print('\nObservation: Higher max_tokens increases quality but also cost.')
print('Temperature 0.7 may add creativity but reduces consistency.')


COST-QUALITY TRADE-OFF ANALYSIS
                   Config Quality Latency  Tokens      Cost  Response Length
Fast/Cheap (T=0, 200 tok)     4/5  2992ms     236 $0.000125             1066
  Standard (T=0, 500 tok)     4/5  7753ms     536 $0.000305             2702
Creative (T=0.7, 500 tok)     4/5  7373ms     536 $0.000305             2704

Observation: Higher max_tokens increases quality but also cost.
Temperature 0.7 may add creativity but reduces consistency.


---
## Demo 3: Day 2 Preview — From Raw API Calls to LangChain

Today we used raw OpenAI API calls. On Day 2, we will use **LangChain** to compose reusable pipelines. This demo shows the same task done both ways, previewing the abstraction power of chains.

In [20]:
# Demo 3: Day 2 Preview — Raw API vs LangChain Pattern

# --- Approach A: Raw API (what we did today) ---
print('APPROACH A: Raw OpenAI API (Day 1 style)')
print('=' * 50)

system_msg = 'You are a McKinsey strategy consultant. Be concise and structured.'
user_msg = 'List 3 key risks in a cross-border pharmaceutical acquisition.'

response = client.chat.completions.create(
    model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
    messages=[
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_msg}
    ],
    temperature=0, max_tokens=300
)
print(response.choices[0].message.content)

# --- Approach B: LangChain pattern (Day 2 preview) ---
print(f'\n{"=" * 50}')
print('APPROACH B: LangChain LCEL Chain (Day 2 style)')
print('=' * 50)

try:
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser

    # Same task, but composable and reusable
    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a McKinsey {practice} consultant. Be concise and structured.'),
        ('user', '{question}')
    ])

    llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'), temperature=0)
    chain = prompt | llm | StrOutputParser()

    # Reusable across different practices and questions
    result = chain.invoke({'practice': 'strategy', 'question': user_msg})
    print(result)
    
    print('\nKey advantage: The chain is REUSABLE with different variables.')
    result2 = chain.invoke({'practice': 'operations', 'question': 'What are 3 supply chain optimization levers for a CPG company?'})
    print(f'\nOperations example: {result2[:150]}...')

except ImportError:
    print('LangChain not installed. Run: pip install langchain langchain-openai')
    print('You will set this up fully on Day 2.')

print('\n' + '=' * 50)
print('Day 2 will cover: LCEL chains, custom tools, StateGraph workflows, multi-agent systems')


APPROACH A: Raw OpenAI API (Day 1 style)
1. **Regulatory Compliance Risks**: Different countries have varying regulations regarding drug approval, pricing, and marketing. Non-compliance can lead to fines, delays, or even the inability to operate in the market.

2. **Cultural Integration Challenges**: Merging distinct corporate cultures can lead to employee dissatisfaction, loss of key talent, and reduced operational efficiency. Misalignment in values and practices may hinder successful integration.

3. **Market Access and Reimbursement Issues**: Understanding local market dynamics, including pricing strategies and reimbursement policies, is crucial. Failure to navigate these can result in poor market penetration and financial performance post-acquisition.

APPROACH B: LangChain LCEL Chain (Day 2 style)
1. **Regulatory Compliance Risks**: Different countries have varying regulations regarding drug approval, manufacturing practices, and marketing. Non-compliance can lead to delays, fines

---
## Task 1: Build an End-to-End Consulting Deliverable Generator

Combine everything from Day 1 into a function that takes a client question, generates a structured consulting response using role-based prompting, auto-evaluates quality, and returns a full deliverable package with metadata.

| Difficulty | Intermediate |
|---|---|


In [21]:
# Task 1 - SOLUTION: Build an End-to-End Consulting Deliverable Generator
#
# Approach: Orchestrate persona selection, structured generation,
# auto-evaluation, and cost tracking into a single deliverable function.

def generate_deliverable(question, practice='Strategy', review=True):
    """Generate a consulting deliverable with auto-evaluation.
    
    Args:
        question: Client question
        practice: McKinsey practice area for persona selection
        review: Whether to run auto-evaluation
    
    Returns:
        dict with content, scores, cost, metadata
    """
    personas = {
        'Strategy': 'You are a McKinsey Strategy & Corporate Finance partner. Use frameworks like Porter\'s Five Forces, Three Horizons, and MECE decomposition. Provide data-driven, board-ready analysis.',
        'Operations': 'You are a McKinsey Operations partner specializing in supply chain, lean, and operational excellence. Quantify improvement opportunities with specific KPIs.',
        'Digital': 'You are a McKinsey Digital partner. Focus on technology-enabled transformation, AI/ML use cases, and digital operating models. Reference industry benchmarks.',
        'M&A': 'You are a McKinsey M&A and PE partner. Focus on due diligence, synergy estimation, integration planning, and value creation. Use deal-specific frameworks.',
    }
    
    system_prompt = personas.get(practice, personas['Strategy'])
    
    # Generate
    start = time.time()
    response = client.chat.completions.create(
        model=os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'),
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': question}
        ],
        temperature=0, max_tokens=600
    )
    latency = round((time.time() - start) * 1000)
    content = response.choices[0].message.content
    usage = response.usage
    
    # Cost estimation
    model = os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini')
    cost = 0
    if model in PRICING:
        cost = usage.prompt_tokens * PRICING[model]['input'] + usage.completion_tokens * PRICING[model]['output']
    
    deliverable = {
        'question': question,
        'practice': practice,
        'content': content,
        'latency_ms': latency,
        'tokens': usage.total_tokens,
        'cost': round(cost, 6),
        'scores': {},
        'verdict': 'NOT EVALUATED'
    }
    
    # Auto-evaluate
    if review:
        criteria = {
            'Strategic Relevance': 'Analysis directly addresses the client question with strategic insight.',
            'MECE Structure': 'Problem is decomposed into non-overlapping, exhaustive categories.',
            'Actionability': 'Recommendations are specific, prioritized, and implementable.',
        }
        total = 0
        for crit_name, crit_desc in criteria.items():
            result = g_eval_quick(question, content, crit_name, crit_desc)
            score = result.get('score', 0)
            deliverable['scores'][crit_name] = score
            total += score
        avg = round(total / len(criteria), 1)
        deliverable['verdict'] = 'PARTNER-READY' if avg >= 4.0 else 'NEEDS REVISION' if avg >= 3.0 else 'REWORK'
        deliverable['avg_score'] = avg
    
    # Print report
    print(f"CONSULTING DELIVERABLE")
    print(f"{'=' * 60}")
    print(f"Practice: {practice} | Latency: {latency}ms | Cost: ${cost:.6f}")
    print(f"\nQ: {question}")
    print(f"\n{content[:400]}{'...' if len(content) > 400 else ''}")
    if review:
        print(f"\nQuality Scores:")
        for k, v in deliverable['scores'].items():
            bar = '\u2588' * v + '\u2591' * (5 - v)
            print(f"  {k:22s} {bar} {v}/5")
        print(f"  Verdict: {deliverable['verdict']} ({avg}/5)")
    
    return deliverable


# Test across practices
d1 = generate_deliverable('How should a luxury retailer enter the Middle East market?', 'Strategy')
print()
d2 = generate_deliverable('What are the Day-1 integration priorities for a healthcare acquisition?', 'M&A')


CONSULTING DELIVERABLE
Practice: Strategy | Latency: 9830ms | Cost: $0.000369

Q: How should a luxury retailer enter the Middle East market?

Entering the Middle East market as a luxury retailer requires a comprehensive strategy that considers the unique characteristics of the region, competitive dynamics, and consumer behavior. Below is a structured analysis using frameworks such as Porter's Five Forces, the Three Horizons framework, and MECE decomposition.

### 1. Market Analysis Using Porter's Five Forces

**a. Threat of New Entrants...

Quality Scores:
  Strategic Relevance    █████ 5/5
  MECE Structure         ████░ 4/5
  Actionability          ████░ 4/5
  Verdict: PARTNER-READY (4.3/5)

CONSULTING DELIVERABLE
Practice: M&A | Latency: 9726ms | Cost: $0.000368

Q: What are the Day-1 integration priorities for a healthcare acquisition?

In a healthcare acquisition, Day-1 integration priorities are critical to ensure a smooth transition and to begin realizing value from the deal. Her

---
## Task 2: Build a Multi-Persona Comparison for a Client Question

A real McKinsey engagement involves multiple expert perspectives. Build a function that generates responses from different practice-area personas for the same question, evaluates each, and recommends which perspective is strongest.

| Difficulty | Intermediate |
|---|---|


In [22]:
# Task 2 - SOLUTION: Build a Multi-Persona Comparison
#
# Approach: Run the same question through multiple personas,
# score each with g_eval, and rank to find the strongest perspective.

def compare_perspectives(question, practices=None):
    """Compare responses from different McKinsey practice perspectives."""
    if practices is None:
        practices = ['Strategy', 'Operations', 'Digital', 'M&A']
    
    print(f"MULTI-PERSONA COMPARISON")
    print(f"{'=' * 60}")
    print(f"Q: {question}\n")
    
    results = []
    for practice in practices:
        d = generate_deliverable(question, practice, review=True)
        results.append({
            'Practice': practice,
            'Avg Score': d.get('avg_score', 0),
            'Verdict': d['verdict'],
            'Tokens': d['tokens'],
            'Cost': f"${d['cost']:.6f}",
        })
        print()
    
    # Rank and recommend
    df = pd.DataFrame(results).sort_values('Avg Score', ascending=False)
    print(f"\n{'=' * 60}")
    print(f"RANKING:")
    print(df.to_string(index=False))
    
    best = df.iloc[0]
    print(f"\nRecommendation: Lead with {best['Practice']} perspective (score: {best['Avg Score']}/5)")
    return df


comparison = compare_perspectives(
    'How should a regional hospital network respond to declining inpatient volumes and rising outpatient demand?'
)


MULTI-PERSONA COMPARISON
Q: How should a regional hospital network respond to declining inpatient volumes and rising outpatient demand?

CONSULTING DELIVERABLE
Practice: Strategy | Latency: 9281ms | Cost: $0.000369

Q: How should a regional hospital network respond to declining inpatient volumes and rising outpatient demand?

To address the challenges of declining inpatient volumes and rising outpatient demand, a regional hospital network can adopt a strategic approach using several frameworks, including Porter's Five Forces, the Three Horizons framework, and MECE (Mutually Exclusive, Collectively Exhaustive) decomposition. Below is a structured analysis and recommendations.

### 1. **Porter's Five Forces Analysis**

*...

Quality Scores:
  Strategic Relevance    █████ 5/5
  MECE Structure         ████░ 4/5
  Actionability          ████░ 4/5
  Verdict: PARTNER-READY (4.3/5)

CONSULTING DELIVERABLE
Practice: Operations | Latency: 8398ms | Cost: $0.000368

Q: How should a regional hospit

---
## Task 3: Build a Consulting AI Readiness Assessment

Before deploying an AI system for consulting work, you need to assess its readiness across multiple dimensions. Build a function that runs a test suite of consulting questions and produces a readiness scorecard.

| Difficulty | Advanced |
|---|---|


In [23]:
# Task 3 - SOLUTION: Build a Consulting AI Readiness Assessment
#
# Approach: Define a test suite of consulting questions across different
# categories, run the model on each, evaluate with g_eval,
# and produce a readiness scorecard with pass/fail per category.

def readiness_assessment(model_config=None):
    """Run a comprehensive readiness assessment for consulting AI deployment."""
    if model_config is None:
        model_config = {'model': os.getenv('OPENAI_MODEL_NAME', 'gpt-4o-mini'), 'temperature': 0}
    
    # Test suite: different consulting task types
    test_suite = {
        'Market Sizing': 'How large is the addressable market for electric vehicle charging infrastructure in Germany by 2030?',
        'Due Diligence': 'What are the top 5 risks a PE firm should evaluate when acquiring a mid-market SaaS company?',
        'Cost Optimization': 'What are the key levers to reduce operating costs by 15% for a national grocery chain?',
        'Strategy': 'Should a regional US bank pursue an acquisition strategy or organic digital transformation to compete with fintechs?',
    }
    
    print(f"CONSULTING AI READINESS ASSESSMENT")
    print(f"{'=' * 60}")
    print(f"Model: {model_config['model']} | Temperature: {model_config['temperature']}")
    print(f"Test cases: {len(test_suite)}\n")
    
    category_scores = {}
    total_tokens = 0
    total_cost = 0
    
    for category, question in test_suite.items():
        response = client.chat.completions.create(
            model=model_config['model'],
            temperature=model_config['temperature'],
            messages=[
                {'role': 'system', 'content': 'You are a McKinsey senior consultant. Provide structured, MECE, data-driven analysis.'},
                {'role': 'user', 'content': question}
            ],
            max_tokens=400
        )
        content = response.choices[0].message.content
        total_tokens += response.usage.total_tokens
        
        # Score on two key criteria
        s1 = g_eval_quick(question, content, 'Quality', 'Response is structured, accurate, and consulting-ready.')['score']
        s2 = g_eval_quick(question, content, 'Actionability', 'Recommendations are specific and implementable.')['score']
        avg = (s1 + s2) / 2
        
        status = 'PASS' if avg >= 3.5 else 'FAIL'
        category_scores[category] = {'score': avg, 'status': status}
        bar = '\u2588' * round(avg) + '\u2591' * (5 - round(avg))
        print(f"  {category:20s} {bar} {avg:.1f}/5 [{status}]")
    
    # Overall readiness
    overall = np.mean([v['score'] for v in category_scores.values()])
    passed = sum(1 for v in category_scores.values() if v['status'] == 'PASS')
    total = len(category_scores)
    
    print(f"\n{'=' * 60}")
    print(f"Overall Score: {overall:.1f}/5.0")
    print(f"Categories Passed: {passed}/{total}")
    print(f"Total Tokens: {total_tokens}")
    
    if passed == total and overall >= 4.0:
        print(f"\nVERDICT: READY FOR DEPLOYMENT")
    elif passed >= total * 0.75:
        print(f"\nVERDICT: CONDITIONAL — needs improvement in failing categories")
    else:
        print(f"\nVERDICT: NOT READY — significant quality gaps")
    
    return category_scores


scores = readiness_assessment()


CONSULTING AI READINESS ASSESSMENT
Model: gpt-4o-mini | Temperature: 0
Test cases: 4

  Market Sizing        ████░ 4.0/5 [PASS]
  Due Diligence        ████░ 4.5/5 [PASS]
  Cost Optimization    █████ 5.0/5 [PASS]
  Strategy             ████░ 3.5/5 [PASS]

Overall Score: 4.2/5.0
Categories Passed: 4/4
Total Tokens: 1799

VERDICT: READY FOR DEPLOYMENT


---
## Task 4: Design a Consulting AI System Architecture (On Paper)

This is a design exercise that previews Day 2. Using everything you learned today, design an AI system for a McKinsey use case. Document your design as a structured specification.

| Difficulty | Advanced |
|---|---|


In [24]:
# Task 4 - SOLUTION: Design a Consulting AI System Architecture
#
# Approach: Generate a structured system design document that maps
# Day 1 concepts to a practical consulting AI architecture.

system_design = {
    'name': 'McKinsey Engagement AI Assistant',
    'use_case': 'Automated research, analysis, and deliverable drafting for strategy engagements',
    'components': {
        'Input Router (Session 1 - Routing Agent)': {
            'purpose': 'Classify incoming requests and route to the right specialist',
            'technique': 'Zero-shot classification with low temperature (T=0)',
            'day2_upgrade': 'LangGraph conditional edges for stateful routing'
        },
        'Analysis Generator (Session 2 - Prompt Engineering)': {
            'purpose': 'Generate structured consulting analysis',
            'technique': 'Role-based personas + CoT prompting + MECE system prompts',
            'day2_upgrade': 'LangChain LCEL chains with custom consulting tools'
        },
        'Quality Gate (Session 3 - Evaluation)': {
            'purpose': 'Auto-evaluate and flag low-quality outputs before delivery',
            'technique': 'G-Eval scoring on MECE, actionability, executive readiness',
            'day2_upgrade': 'LangGraph cycles for self-correction loops'
        },
        'Knowledge Base (Session 1 - Embeddings)': {
            'purpose': 'Retrieve relevant prior work and industry data',
            'technique': 'Cosine similarity search over document embeddings',
            'day3_upgrade': 'Full RAG pipeline with ChromaDB and reranking'
        },
    },
    'model_selection': {
        'routing': 'gpt-4o-mini (T=0, max_tokens=10) — fast, cheap classification',
        'analysis': 'gpt-4o-mini (T=0, max_tokens=600) — structured, deterministic',
        'creative': 'gpt-4o-mini (T=0.7, max_tokens=600) — brainstorming and ideation',
        'evaluation': 'gpt-4o-mini (T=0, max_tokens=200) — consistent scoring',
    },
    'quality_thresholds': {
        'partner_ready': 'avg score >= 4.0 across all criteria',
        'needs_revision': 'avg score 3.0-3.9 — route to human review',
        'rework': 'avg score < 3.0 — regenerate with different approach',
    },
    'estimated_cost_per_query': '$0.001-0.005 with gpt-4o-mini',
}

print('CONSULTING AI SYSTEM DESIGN')
print('=' * 60)
print(f"System: {system_design['name']}")
print(f"Use Case: {system_design['use_case']}\n")

print('COMPONENTS:')
for comp, details in system_design['components'].items():
    print(f"\n  {comp}")
    for k, v in details.items():
        print(f"    {k}: {v}")

print(f"\nMODEL SELECTION:")
for task, config in system_design['model_selection'].items():
    print(f"  {task:15s}: {config}")

print(f"\nQUALITY THRESHOLDS:")
for level, desc in system_design['quality_thresholds'].items():
    print(f"  {level:15s}: {desc}")

print(f"\nEstimated Cost: {system_design['estimated_cost_per_query']}")
print('\nThis architecture maps directly to Day 2 frameworks:')
print('  - Input Router -> LangGraph conditional edges')
print('  - Analysis Generator -> LangChain LCEL chains')
print('  - Quality Gate -> LangGraph cycles (self-correction)')
print('  - Knowledge Base -> Day 3 RAG pipeline')


CONSULTING AI SYSTEM DESIGN
System: McKinsey Engagement AI Assistant
Use Case: Automated research, analysis, and deliverable drafting for strategy engagements

COMPONENTS:

  Input Router (Session 1 - Routing Agent)
    purpose: Classify incoming requests and route to the right specialist
    technique: Zero-shot classification with low temperature (T=0)
    day2_upgrade: LangGraph conditional edges for stateful routing

  Analysis Generator (Session 2 - Prompt Engineering)
    purpose: Generate structured consulting analysis
    technique: Role-based personas + CoT prompting + MECE system prompts
    day2_upgrade: LangChain LCEL chains with custom consulting tools

  Quality Gate (Session 3 - Evaluation)
    purpose: Auto-evaluate and flag low-quality outputs before delivery
    technique: G-Eval scoring on MECE, actionability, executive readiness
    day2_upgrade: LangGraph cycles for self-correction loops

  Knowledge Base (Session 1 - Embeddings)
    purpose: Retrieve relevant prio

---
## Summary

In this capstone session, we integrated all Day 1 concepts:

1. **End-to-End Pipeline** — Combined API calls (Session 1), structured prompting (Session 2), and G-Eval scoring (Session 3) into a single consulting deliverable generator
2. **Cost-Quality Trade-offs** — Analyzed how temperature, max_tokens, and model choice affect quality, latency, and cost
3. **Day 2 Preview** — Compared raw API calls with LangChain LCEL chains to preview the composability advantages

### Key Takeaways
- Real consulting AI systems combine multiple techniques from Sessions 1-3
- Auto-evaluation with G-Eval enables quality gates without human review
- Different practice areas benefit from different personas and configurations
- Cost estimation is critical for scoping AI-augmented engagements
- Day 2 will replace raw API calls with LangChain chains, LangGraph workflows, and multi-agent systems

### What's Next (Day 2)
- **Session 1:** Structured Outputs — JSON mode, function calling, Pydantic
- **Session 2:** LangChain — LCEL chains, custom consulting tools
- **Session 3:** LangGraph — StateGraph, conditional edges, cycles
- **Session 4:** Multi-Agent — supervisor-worker teams modeling McKinsey engagement structures